# **AI Advanced Ahmed Yousrey Course Practice**

---
# **DAY 4 — Customer Behavior Clustering**
---

## STEP 1 — Importing Libraries

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA


---
##STEP 2 - Importing Data

----

In [ ]:
df=pd.read_csv("/content/marketing_campaign.csv" , sep="\t")

---
##STEP 3 - EDA
----

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

### Initial Data Inspection

* The dataset contains **2,240 customer records**.
* `Income` contains **24 missing values**, which will be removed before further analysis.
* Most columns are stored as **integer** data types.
* The main non-integer columns are:

  * `Dt_Customer`
  * `Marital_Status`
  * `Education`
* The dataset contains both numerical and categorical features, so preprocessing and encoding will be required before applying clustering algorithms.


---
##STEP 4 - Preprocessing
----

**1- remove irrelevant data**

`Z_CostContact` and `Z_Revenue` are constant columns (single unique value each) — they carry zero information and would add noise to clustering. `ID` is just a row identifier, not a behavioral feature.

In [ ]:
print(len(df['Z_CostContact'].unique()))
print(len(df['Z_Revenue'].unique()))

In [ ]:
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])

**2-handle missing values**

Only `Income` has 24 missing rows (~1% of the dataset). Dropping them is safe here — imputing income is risky because it could distort the spending patterns that are central to clustering.

In [ ]:
df.dropna(subset=['Income'], inplace=True)

**3-feature engineering**

Three new features are created from existing raw columns:
- `Age` from `Year_Birth` — more interpretable than a birth year
- `Customer_Tenure_Days` from `Dt_Customer` — how many years since the customer enrolled (extracted as year difference)
- `Total_Spending` — sum of all product spend columns; a single high-signal feature that captures overall customer value

In [ ]:
df['Age'] = 2026 - df['Year_Birth']
df=df.drop(columns=['Year_Birth'])

In [ ]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
df['Dt_Customer'] = df['Dt_Customer'].dt.year
df['Dt_Customer'] = 2026 - df['Dt_Customer']

df = df.rename(columns={
    "Dt_Customer": "Customer_Tenure_Days"
})

In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_cols].sum(axis=1)

**4-Encoding**

  * `Marital_Status` — one-hot encoded since there is no ordinal relationship between categories
  * `Education` — ordinal encoded (0–4) since education levels have a natural order

In [ ]:
df['Marital_Status'].unique()

In [ ]:
df = pd.get_dummies(
    df,
    columns=["Marital_Status"],
    dtype=int
)

In [ ]:
df['Education'].unique()

In [ ]:
education_map = {
    "Basic": 0,
    "2n Cycle": 1,
    "Graduation": 2,
    "Master": 3,
    "PhD": 4
}

df["Education"] = df["Education"].map(education_map)

**5-Scale data**

K-Means uses Euclidean distance, so features with large ranges (like `Income` in tens of thousands) would dominate over features with small ranges (like `Kidhome` which is 0–2). `StandardScaler` brings all features to the same scale before clustering.

`Response` is excluded from clustering — it is a target-like label that should not influence how customers are grouped.

In [ ]:
from sklearn.preprocessing import StandardScaler

clustering_features = df.drop(columns=["Response"])

scaler = StandardScaler()

df_scaled = scaler.fit_transform(clustering_features)

df_scaled = pd.DataFrame(
    df_scaled,
    columns=clustering_features.columns,
    index=clustering_features.index
)

In [ ]:
df_scaled.head()

In [ ]:
df_scaled.shape

**6-Check and cap outliers**

Outliers are detected using the IQR method on continuous numeric columns. Rather than dropping outlier rows (which would remove genuine high-spending customers), values are **capped** at the IQR bounds using `clip`. This limits the influence of extreme values without losing any data.

Binary columns (AcceptedCmp1–5, Complain, etc.) and ordinal-encoded features are excluded from capping — they only take values 0 or 1, so IQR flagging on them is not meaningful.

In [ ]:
numeric_cols = [
    "Income",
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
    "Total_Spending",
    "NumDealsPurchases",
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases",
    "NumWebVisitsMonth",
    "Recency"
]

Q1 = clustering_features[numeric_cols].quantile(0.25)
Q3 = clustering_features[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (
    (clustering_features[numeric_cols] < lower_bound) |
    (clustering_features[numeric_cols] > upper_bound)
)

print("Outlier counts per column:")
print(outliers.sum().sort_values(ascending=False))

In [ ]:
# Cap outliers at IQR bounds instead of dropping rows
for col in numeric_cols:
    clustering_features[col] = clustering_features[col].clip(
        lower=lower_bound[col],
        upper=upper_bound[col]
    )

print("Outlier capping applied to:", numeric_cols)

---
##STEP 5 - Clustering
----

**Choosing the number of clusters — Elbow Method**

Inertia (within-cluster sum of squares) is computed for K from 2 to 10. The goal is to find the point where adding more clusters stops producing a significant drop in inertia — the 'elbow'.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.plot(range(2, 11), inertia, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()

**Choosing the number of clusters — Silhouette Score**

The Silhouette Score measures how well each point fits its own cluster versus neighboring clusters. A higher score (closer to 1) means better-defined clusters. This is used alongside the Elbow Method because the elbow plot did not show a sharp bend — the Silhouette Score provides a second signal to confirm the best K.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(df_scaled)

    score = silhouette_score(df_scaled, labels)
    silhouette_scores.append(score)

print(silhouette_scores)

In [ ]:
plt.plot(range(2, 11), silhouette_scores, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.show()

**Choosing K = 2**

The Elbow Method did not produce a sharp bend — the inertia curve declined smoothly. The Silhouette Score was highest at K=2, making it the clearest data-driven choice. K=2 also aligns with a natural business interpretation: customers split into a high-value segment and a low-value segment, which is a common and actionable segmentation for CRM campaigns.

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(df_scaled)

df["Cluster"] = cluster_labels
df["Cluster"].value_counts()

**Cluster profiling — key features**

A targeted profile of the most interpretable behavioral and demographic features per cluster.

In [ ]:
cluster_profile = df.groupby("Cluster")[[
    "Age",
    "Income",
    "Recency",
    "Total_Spending",
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases",
    "NumWebVisitsMonth",
    "Kidhome",
    "Teenhome"
]].mean()

cluster_profile

**Cluster profiling — full feature view**

Full mean values across all features per cluster, transposed for readability. The `Cluster` column is excluded from the profile to avoid it appearing as a feature.

In [ ]:
profile_cols = [col for col in df.columns if col != "Cluster"]

cluster_profile_full = df.groupby("Cluster")[profile_cols].mean(numeric_only=True)

cluster_profile_full.T

**Experiment: removing `Total_Spending` to test if individual spend columns produce better clusters**

`Total_Spending` is a sum of the individual `Mnt*` columns, so it is highly correlated with them. This experiment checks whether removing it changes the optimal K or cluster quality, since the individual columns already carry the same information.

In [ ]:
# Remove Total_Spending
clustering_features_no_total = clustering_features.drop(
    columns=["Total_Spending"]
)

# Scale again
scaler_no_total = StandardScaler()

df_scaled_no_total = scaler_no_total.fit_transform(
    clustering_features_no_total
)

# Test K from 2 to 10
silhouette_scores_no_total = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(df_scaled_no_total)

    score = silhouette_score(
        df_scaled_no_total,
        labels
    )

    silhouette_scores_no_total.append(score)

# Print results
for k, score in zip(range(2, 11), silhouette_scores_no_total):
    print(f"K={k}: {score:.4f}")

# Best K
best_k = range(2, 11)[
    silhouette_scores_no_total.index(
        max(silhouette_scores_no_total)
    )
]

print(f"\nBest K: {best_k}")
print(
    f"Best Silhouette Score: "
    f"{max(silhouette_scores_no_total):.4f}"
)

# Final KMeans using best K
kmeans_no_total = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster_no_total"] = kmeans_no_total.fit_predict(
    df_scaled_no_total
)

# Cluster sizes
print("\nCluster Sizes:")
print(df["Cluster_no_total"].value_counts())

# Cluster profile — exclude cluster label columns from the mean
profile_cols_no_total = [
    col for col in df.columns
    if col not in ["Cluster", "Cluster_no_total"]
]

cluster_profile_no_total = df.groupby(
    "Cluster_no_total"
)[profile_cols_no_total].mean(numeric_only=True)

print("\nCluster Profile:")
display(cluster_profile_no_total.T)

**Campaign response rate by cluster**

Checking whether the clusters differ in how often customers responded to marketing campaigns. A meaningful difference in response rates validates that the clusters capture something behaviorally real.

In [ ]:
response_by_cluster = df.groupby("Cluster_no_total")["Response"].mean()

print(response_by_cluster)


campaign_cols = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Response"
]

campaign_response = df.groupby("Cluster_no_total")[campaign_cols].mean() * 100

display(campaign_response.T)

**Naming the clusters**

Based on the cluster profiles:
- **Cluster 0 — Low-Value Customers**: Lower income, more children at home, low spending across all categories, higher web visit frequency but fewer actual purchases.
- **Cluster 1 — High-Value Customers**: Higher income, fewer children, significantly higher spending across all product categories, more catalog and store purchases, and much higher campaign acceptance rates.

In [ ]:
cluster_names = {
    0: "Low-Value Customers",
    1: "High-Value Customers"
}

**PCA Visualization**

PCA is used here purely for visualization — the clustering was already done on the full scaled feature space. PCA compresses the high-dimensional data into 2 components so the cluster structure can be plotted. The explained variance tells us how much of the original structure is preserved in this 2D view.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import numpy as np

pca = PCA(n_components=2)

pca_result = pca.fit_transform(df_scaled_no_total)

pca_df = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"],
    index=df.index
)

pca_df["Cluster"] = df["Cluster_no_total"]

cluster_names = {
    0: "Low-Value Customers",
    1: "High-Value Customers"
}

plt.figure(figsize=(10, 7))

for cluster in sorted(pca_df["Cluster"].unique()):

    cluster_data = pca_df[
        pca_df["Cluster"] == cluster
    ]

    x = cluster_data["PC1"].values
    y = cluster_data["PC2"].values

    # Plot cluster points
    plt.scatter(
        x,
        y,
        alpha=0.5,
        label=cluster_names[cluster]
    )

    # Create boundary around the cluster
    points = np.column_stack((x, y))

    hull = ConvexHull(points)

    hull_points = points[hull.vertices]

    # Close the boundary
    hull_points = np.vstack([
        hull_points,
        hull_points[0]
    ])

    plt.plot(
        hull_points[:, 0],
        hull_points[:, 1],
        linewidth=2
    )

    # Cluster center label
    center_x = x.mean()
    center_y = y.mean()

    plt.text(
        center_x,
        center_y,
        cluster_names[cluster],
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title("Customer Segmentation using KMeans + PCA")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

print(
    f"PC1: {pca.explained_variance_ratio_[0]:.2%}"
)

print(
    f"PC2: {pca.explained_variance_ratio_[1]:.2%}"
)

print(
    f"Total explained variance: "
    f"{pca.explained_variance_ratio_.sum():.2%}"
)